# 03 Fine-Tune Qwen7B for Report Generation

This notebook performs a lightweight QLoRA supervised fine-tuning run for report-style generation.

Dependency order:

1. Run **Mount Drive**.
2. Run **Install and Import**.
3. Run **Configure Paths and Hyperparameters**.
4. Run **Build Fine-Tuning Dataset**.
5. Run **Load Qwen7B with QLoRA** on a GPU runtime.
6. Run **Fine-Tune and Save Adapter**.

The base model is not saved to GitHub. The LoRA adapter is saved to Google Drive.

## Mount Drive

Dependency: none.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install and Import

Dependency: Drive is mounted. This installs the QLoRA training stack.

In [ ]:
%pip install -q "transformers>=4.51.0" datasets accelerate bitsandbytes peft trl pandas

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset, concatenate_datasets, load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

## Configure Paths and Hyperparameters

Dependency: imports are ready.

Default dataset choice: `GEM/totto`, an open table-to-text dataset. This is not an ML-report dataset, but it is useful for report generation because it trains the model to convert structured evidence into faithful text.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CarDD_YOLO11')
REPORTS_ROOT = DRIVE_ROOT / 'reports'
TRAIN_ROOT = DRIVE_ROOT / 'runs' / 'train' / 'yolo11n_seg'
DEMO_ROOT = DRIVE_ROOT / 'runs' / 'predict' / 'demo'
LLM_ROOT = DRIVE_ROOT / 'llm_adapters'
ADAPTER_DIR = LLM_ROOT / 'qwen2_5_7b_cardd_report_lora'
SFT_DATA_JSONL = REPORTS_ROOT / 'qwen7b_report_sft_data.jsonl'
SFT_META_JSON = REPORTS_ROOT / 'qwen7b_report_sft_metadata.json'

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
OPEN_DATASET_ID = 'GEM/totto'
OPEN_TRAIN_EXAMPLES = 1200
OPEN_EVAL_EXAMPLES = 120

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 1
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
SAVE_STEPS = 100
SEED = 42

REPORTS_ROOT.mkdir(parents=True, exist_ok=True)
LLM_ROOT.mkdir(parents=True, exist_ok=True)
print('Adapter output:', ADAPTER_DIR)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Build Fine-Tuning Dataset

Dependency: paths are configured. This cell downloads a small slice of `GEM/totto` and adds a few CarDD-specific report-format examples from existing metrics.

Output: `reports/qwen7b_report_sft_data.jsonl` in Drive.

In [ ]:
def format_messages(system, user, assistant):
    return [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': assistant},
    ]

SYSTEM_PROMPT = (
    'You write faithful, concise technical report text from structured evidence. '
    'Use only the provided evidence and do not invent numbers or claims.'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

totto_train = load_dataset(OPEN_DATASET_ID, split=f'train[:{OPEN_TRAIN_EXAMPLES}]')
totto_eval = load_dataset(OPEN_DATASET_ID, split=f'validation[:{OPEN_EVAL_EXAMPLES}]')

def to_totto_messages(example):
    user = (
        'Convert the following structured evidence into a concise factual report sentence.\n\n'
        f"Evidence:\n{example['linearized_input']}"
    )
    return {'messages': format_messages(SYSTEM_PROMPT, user, example['target'])}

train_dataset = totto_train.map(to_totto_messages, remove_columns=totto_train.column_names)
eval_dataset = totto_eval.map(to_totto_messages, remove_columns=totto_eval.column_names)

test_metrics_path = REPORTS_ROOT / 'yolo11n_seg_test_metrics.json'
run_summary_path = REPORTS_ROOT / 'yolo11n_seg_run_summary.json'
results_csv = TRAIN_ROOT / 'results.csv'

cardd_examples = []
if test_metrics_path.exists() and run_summary_path.exists() and results_csv.exists():
    metrics = json.loads(test_metrics_path.read_text(encoding='utf-8'))
    run_summary = json.loads(run_summary_path.read_text(encoding='utf-8'))
    final_row = pd.read_csv(results_csv).iloc[-1].to_dict()

    evidence = json.dumps({
        'model': 'YOLO11n-seg',
        'dataset': 'CarDD',
        'epochs': 100,
        'test_metrics': metrics,
        'artifacts': run_summary,
    }, indent=2)
    response = (
        'The CarDD reproduction fine-tuned YOLO11n-seg for 100 epochs and produced a complete detection and instance segmentation pipeline. '
        f"On the test split, box mAP50 was {metrics['metrics/mAP50(B)']:.3f} and mask mAP50 was {metrics['metrics/mAP50(M)']:.3f}. "
        f"The final Drive artifacts include best.pt, last.pt, and the exported ONNX model."
    )
    cardd_examples.append({'messages': format_messages(SYSTEM_PROMPT, 'Write a concise project result paragraph from this evidence:\n\n' + evidence, response)})

    class_evidence = json.dumps({
        'per_class_test_map50': [
            {'class': 'dent', 'box_map50': 0.492, 'mask_map50': 0.496},
            {'class': 'scratch', 'box_map50': 0.511, 'mask_map50': 0.475},
            {'class': 'crack', 'box_map50': 0.211, 'mask_map50': 0.219},
            {'class': 'glass shatter', 'box_map50': 0.978, 'mask_map50': 0.978},
            {'class': 'lamp broken', 'box_map50': 0.790, 'mask_map50': 0.778},
            {'class': 'tire flat', 'box_map50': 0.882, 'mask_map50': 0.882},
        ]
    }, indent=2)
    class_response = (
        'Per-class results show that glass shatter, tire flat, and lamp broken are the strongest categories, '
        'while crack, scratch, and dent are harder because their visual evidence is finer and more ambiguous.'
    )
    cardd_examples.append({'messages': format_messages(SYSTEM_PROMPT, 'Write a per-class analysis from this evidence:\n\n' + class_evidence, class_response)})

    demo_evidence = '\n'.join([
        'image0: 1 scratch',
        'image1: no detections',
        'image2: 1 lamp broken',
        'image3: 1 dent, 1 scratch',
        'image4: 1 scratch',
        'image5: 2 cracks',
        'image6: 1 scratch',
        'image7: no detections',
    ])
    demo_response = (
        'The demo notebook sampled eight test images and generated visual predictions with class labels, confidence scores, boxes, and masks. '
        'Six images produced saved label files, while two images had no detections under the selected confidence threshold.'
    )
    cardd_examples.append({'messages': format_messages(SYSTEM_PROMPT, 'Summarize this demo inference evidence:\n\n' + demo_evidence, demo_response)})

if cardd_examples:
    train_dataset = concatenate_datasets([train_dataset, Dataset.from_list(cardd_examples)])

def add_text(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False)}

train_dataset = train_dataset.map(add_text)
eval_dataset = eval_dataset.map(add_text)

with SFT_DATA_JSONL.open('w', encoding='utf-8') as f:
    for row in train_dataset:
        f.write(json.dumps({'messages': row['messages']}, ensure_ascii=False) + '\n')

metadata = {
    'model_id': MODEL_ID,
    'open_dataset': OPEN_DATASET_ID,
    'open_train_examples': OPEN_TRAIN_EXAMPLES,
    'open_eval_examples': OPEN_EVAL_EXAMPLES,
    'cardd_examples': len(cardd_examples),
    'train_rows': len(train_dataset),
    'eval_rows': len(eval_dataset),
    'max_seq_length': MAX_SEQ_LENGTH,
}
SFT_META_JSON.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(json.dumps(metadata, indent=2))

## Load Qwen7B with QLoRA

Dependency: fine-tuning dataset is built. Use a GPU runtime, preferably Colab L4 or better.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

print('Model loaded for QLoRA training.')

## Fine-Tune and Save Adapter

Dependency: model is loaded with QLoRA. This runs a small supervised fine-tuning job and saves only the LoRA adapter to Drive.

In [ ]:
training_args = SFTConfig(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    dataset_text_field='text',
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
)

trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

adapter_meta = json.loads(SFT_META_JSON.read_text(encoding='utf-8'))
adapter_meta['adapter_dir'] = str(ADAPTER_DIR)
adapter_meta['status'] = 'complete'
SFT_META_JSON.write_text(json.dumps(adapter_meta, indent=2), encoding='utf-8')

print('Adapter saved:', ADAPTER_DIR)
print('Metadata:', SFT_META_JSON)